# PyDI Data Integration Workflow: Videogames

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with vidoegame datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
  - [Step 1: Load Target Schema and Normalization Spec](#step-1-load-target-schema-and-normalization-spec)
  - [Step 2: Load Source Datasets](#step-2-load-source-datasets)
  - [Step 3: LLM-Based Schema Matching](#step-3-llm-based-schema-matching)
  - [Step 4: Schema Matching Evaluation](#step-4-evaluate-schema-matching-against-gold-mapping)
  - [Step 5: Translate and Normalize](#step-5-translate-and-normalize)
- [Part 2: Data Profiling](#part-2-data-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

### Datasets

- **DBpedia**: 65,000 records
- **Metacritic**: 20,494 records
- **Global Sales Ranking**: 7,877 records

## Part 1: Schema Matching and Value Normalization

In [1]:
import time
start_time = time.time()

from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaMappingEvaluator, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

/Users/aaronsteiner/Documents/GitHub/PyDI/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Target Schema and Normalization Spec

In [3]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")

# Set genres column manually to list type
spec.set_column("genres", output_type="list")

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,releaseYear,datetime
3,developer,string
4,publisher,string
5,platform,string
6,criticScore,float
7,userScore,float
8,ESRB,string
9,series,string


## Step 2: Load Source Datasets

In [4]:
from PyDI.io import load_xml, load_csv, load_json
dbpedia = load_csv(INPUT_DIR / "data" / "dbpedia.csv")
dbpedia.attrs["dataset_name"] = "dbpedia"
dbpedia.head()

,wiki_ref,title,launch_yr,studio,system,genre,franchise
0,dbpedia_1,San Francisco Rush 2049,2006-01-01,Handheld Games,Game Boy Color,Racing video game,Rush (video game series)
1,dbpedia_2,RoboCop (1988 video game),1989-01-01,Ocean Software,Arcade video game,Beat 'em up,List of RoboCop video games
2,dbpedia_3,Air (video game),2016-01-01,Key (company),PlayStation Vita,Eroge,NaN
3,dbpedia_4,Fallout 2,1998-01-01,Black Isle Studios,Mac OS X,Role-playing video game,Fallout (series)
4,dbpedia_5,SpongeBob SquarePants: Creature from the Krust...,2006-01-01,Blitz Games,Wii,Platform game,SpongeBob SquarePants video games


In [5]:
metacritic = load_csv(INPUT_DIR / "data" / "metacritic.csv")
metacritic.attrs["dataset_name"] = "metacritic"
metacritic.head()

,mc_id,game_title,year_published,made_by,console,genres,press_rating,player_rating,age_rating
0,metacritic_1,Red Dead Redemption 2,2018-01-01,Rockstar Games,Xbox One,"Action Adventure,Open-World",97.0,8.3,M
1,metacritic_2,Grand Theft Auto IV,2008-01-01,Rockstar North,Xbox 360,"Action Adventure,Modern,Modern,Open-World",98.0,8.0,M
2,metacritic_3,SoulCalibur,1999-01-01,Namco,Dreamcast,"Action,Fighting,3D",98.0,8.4,T
3,metacritic_4,Tony Hawk's Pro Skater 2,2000-01-01,Neversoft Entertainment,PlayStation,"Sports,Alternative,Skateboarding",98.0,7.5,T
4,metacritic_5,Super Mario Galaxy,2007-01-01,Nintendo,Wii,"Action,Platformer,Platformer,3D,3D",97.0,9.1,E


In [6]:
sales = load_csv(INPUT_DIR / "data" / "sales.csv")
sales.attrs["dataset_name"] = "sales"
sales.head()

,rec_id,prod_title,launch_dt,studio,dist,hw,genre,press_score,comm_rating,age_classification,units_sold_mm
0,sales_1,Wii Sports,2006-01-01,Nintendo,Nintendo,Wii,Sports,76,8.0,E,82
1,sales_2,Mario Kart Wii,2008-01-01,Nintendo,Nintendo,Wii,Racing,82,8.3,E,35
2,sales_3,Wii Sports Resort,2009-01-01,Nintendo,Nintendo,Wii,Sports,80,8.0,E,32
3,sales_4,New Super Mario Bros.,2006-01-01,Nintendo,Nintendo,DS,Platform,89,8.5,E,29
4,sales_5,Wii Play,2006-01-01,Nintendo,Nintendo,Wii,Misc,58,6.6,E,28


## Step 3: LLM-Based Schema Matching

In [7]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match dbpedia dataset
dbpedia_mapping = matcher.match(dbpedia, df_target)

dbpedia_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,dbpedia,wiki_ref,target_schema,id,0.95,llm_based_matching
1,dbpedia,title,target_schema,name,0.95,llm_based_matching
2,dbpedia,launch_yr,target_schema,releaseYear,0.95,llm_based_matching
3,dbpedia,studio,target_schema,developer,0.95,llm_based_matching
4,dbpedia,system,target_schema,platform,0.95,llm_based_matching
5,dbpedia,genre,target_schema,genres,0.95,llm_based_matching
6,dbpedia,franchise,target_schema,series,0.95,llm_based_matching


In [8]:
metacritic_mapping = matcher.match(metacritic, df_target)
metacritic_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,metacritic,mc_id,target_schema,id,0.95,llm_based_matching
1,metacritic,game_title,target_schema,name,0.95,llm_based_matching
2,metacritic,year_published,target_schema,releaseYear,0.95,llm_based_matching
3,metacritic,made_by,target_schema,developer,0.95,llm_based_matching
4,metacritic,console,target_schema,platform,0.95,llm_based_matching
5,metacritic,genres,target_schema,genres,0.95,llm_based_matching
6,metacritic,press_rating,target_schema,criticScore,0.95,llm_based_matching
7,metacritic,player_rating,target_schema,userScore,0.95,llm_based_matching
8,metacritic,age_rating,target_schema,ESRB,0.95,llm_based_matching


In [9]:
sales_mapping = matcher.match(sales, df_target)
sales_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,sales,rec_id,target_schema,id,0.95,llm_based_matching
1,sales,prod_title,target_schema,name,0.95,llm_based_matching
2,sales,launch_dt,target_schema,releaseYear,0.95,llm_based_matching
3,sales,studio,target_schema,developer,0.95,llm_based_matching
4,sales,dist,target_schema,publisher,0.95,llm_based_matching
5,sales,hw,target_schema,platform,0.95,llm_based_matching
6,sales,genre,target_schema,genres,0.95,llm_based_matching
7,sales,press_score,target_schema,criticScore,0.95,llm_based_matching
8,sales,comm_rating,target_schema,userScore,0.95,llm_based_matching
9,sales,age_classification,target_schema,ESRB,0.95,llm_based_matching


## Step 4: Evaluate Schema Matching Against Gold Mapping


In [10]:
# Load the manually curated schema-matching gold standard
with open(INPUT_DIR / "schemamatching" / "sm_mapping_gold.json") as f:
    schema_mapping_gold = json.load(f)

schema_mapping_gold_df = pd.DataFrame(schema_mapping_gold["mappings"])
schema_mapping_predictions = pd.concat(
    [dbpedia_mapping, metacritic_mapping, sales_mapping],
    ignore_index=True,
)

schema_matching_metrics = SchemaMappingEvaluator.evaluate(
    schema_mapping_predictions,
    schema_mapping_gold_df,
    complete=True,
)

schema_matching_summary = pd.DataFrame([
    {
        **schema_matching_metrics,
        "predicted_mappings": len(schema_mapping_predictions),
        "gold_mappings": len(schema_mapping_gold_df),
    }
])

per_source_schema_matching = pd.DataFrame([
    {
        "source_dataset": source_dataset,
        **SchemaMappingEvaluator.evaluate(
            schema_mapping_predictions[
                schema_mapping_predictions["source_dataset"] == source_dataset
            ],
            schema_mapping_gold_df[
                schema_mapping_gold_df["source_dataset"] == source_dataset
            ],
            complete=True,
        ),
    }
    for source_dataset in sorted(schema_mapping_gold_df["source_dataset"].unique())
])

display(schema_matching_summary)
per_source_schema_matching


,precision,recall,f1,correct,matched,correct_total,missing,predicted_mappings,gold_mappings
0,1.0,1.0,1.0,26,26,26,0,26,26


,source_dataset,precision,recall,f1,correct,matched,correct_total,missing
0,dbpedia,1.0,1.0,1.0,7,7,7,0
1,metacritic,1.0,1.0,1.0,9,9,9,0
2,sales,1.0,1.0,1.0,10,10,10,0


## Step 5: Translate and Normalize


In [11]:
# Unify platform names across datasets
platform_groups = {
    "Nintendo Entertainment System": ["NES"],
    "Super Nintendo": ["SNES", "Super Nintendo Entertainment System"],
    "Nintendo 64": ["N64"],
    "GameCube": ["Nintendo GameCube", "GC"],
    "Wii": ["Nintendo Wii"],
    "Game Boy Color": ["GBC"],
    "Game Boy Advance": ["GBA"],
    "Game Boy": ["GB"],
    "DS": ["Nintendo DS"],
    "3DS": ["Nintendo 3DS"],
    "Switch": ["Nintendo Switch"],

    "Playstation": [
        "Playstation (console)", "PlayStation (console)",
        "Playstation 1", "PlayStation 1",
        "PS1", "PSX", "PS"
    ],

    "PS2": ["Playstation 2", "PlayStation 2"],
    "PS3": ["Playstation 3", "PlayStation 3"],
    "PS4": ["Playstation 4", "PlayStation 4"],
    "Playstation Portable": ["PSP"],
    "Playstation Vita": ["PSV", "PS Vita"],
    "Playstation VR": ["PSVR", "PS VR"],

    "Xbox": ["XB", "Xbox (console)"],
    "Xbox One": ["XOne"],
    "Xbox 360": ["X360"],

    "PC": ["Microsoft Windows", "Windows"],
}

platform_map = {}
for canonical, variants in platform_groups.items():
    for alias in variants:
        platform_map[alias.lower()] = canonical

def normalize_platform(series):
    def _norm(x):
        # Leave missing or non-string values as they are
        if not isinstance(x, str):
            return x
        key = x.strip().lower()
        return platform_map.get(key, x.strip())
    
    return series.apply(_norm)

dbpedia["system"] = normalize_platform(dbpedia["system"])
metacritic["console"] = normalize_platform(metacritic["console"])
sales["hw"] = normalize_platform(sales["hw"])

In [12]:
# Create id columns based on index (starting with 1)
sales["id"] = sales.index + 1
metacritic["id"] = metacritic.index + 1
dbpedia["id"] = dbpedia.index + 1

# Make id entries more explicit
dbpedia["id"] = dbpedia["id"].apply(lambda x: f"dbpedia_{x}")
metacritic["id"] = metacritic["id"].apply(lambda x: f"metacritic_{x}")
sales["id"] = sales["id"].apply(lambda x: f"sales_{x}")

In [13]:
# Clean up noisy dbpedia dataset

# Remove suffix (" (video game)") from franchise
dbpedia_cleaned = dbpedia.copy()
dbpedia_cleaned["franchise"] = dbpedia_cleaned["franchise"].str.replace(r" \(video game\)$", "", regex=True)


In [14]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("releaseYear", output_type="datetime")
dbpedia_normalized = translator.translate(
    dbpedia_cleaned, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse special date format (e.g., Oct 26, 2018)
spec.set_column("releaseYear", output_type="datetime", date_format="%b %d, %Y")

metacritic_normalized = translator.translate(
    metacritic, metacritic_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse year-only values (e.g., "1929", "2010")
spec.set_column("releaseYear", output_type="datetime", date_format="%Y")

sales_normalized = translator.translate(
    sales, sales_mapping,
    normalize=spec, on_failure="keep"
)

In [15]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head()

,id,name,releaseYear,developer,platform,series,genres
0,dbpedia_1,San Francisco Rush 2049,2006-01-01,Handheld Games,Game Boy Color,Rush (video game series),Racing video game
1,dbpedia_2,RoboCop (1988 video game),1989-01-01,Ocean Software,Arcade video game,List of RoboCop video games,Beat 'em up
2,dbpedia_3,Air (video game),2016-01-01,Key (company),PlayStation Vita,NaN,Eroge
3,dbpedia_4,Fallout 2,1998-01-01,Black Isle Studios,Mac OS X,Fallout (series),Role-playing video game
4,dbpedia_5,SpongeBob SquarePants: Creature from the Krust...,2006-01-01,Blitz Games,Wii,SpongeBob SquarePants video games,Platform game


In [16]:
metacritic_cols = [c for c in target_columns if c in metacritic_normalized.columns]
metacritic_normalized[metacritic_cols].head()

,id,name,releaseYear,developer,platform,criticScore,userScore,ESRB,genres
0,metacritic_1,Red Dead Redemption 2,2018-01-01,Rockstar Games,Xbox One,97.0,8.3,M,"Action Adventure,Open-World"
1,metacritic_2,Grand Theft Auto IV,2008-01-01,Rockstar North,Xbox 360,98.0,8.0,M,"Action Adventure,Modern,Modern,Open-World"
2,metacritic_3,SoulCalibur,1999-01-01,Namco,Dreamcast,98.0,8.4,T,"Action,Fighting,3D"
3,metacritic_4,Tony Hawk's Pro Skater 2,2000-01-01,Neversoft Entertainment,PlayStation,98.0,7.5,T,"Sports,Alternative,Skateboarding"
4,metacritic_5,Super Mario Galaxy,2007-01-01,Nintendo,Wii,97.0,9.1,E,"Action,Platformer,Platformer,3D,3D"


In [17]:
sales_cols = [c for c in target_columns if c in sales_normalized.columns]
sales_normalized[sales_cols].head()

,id,name,releaseYear,developer,publisher,platform,criticScore,userScore,ESRB,genres
0,sales_1,Wii Sports,2006-01-01,Nintendo,Nintendo,Wii,76.0,8.0,E,Sports
1,sales_2,Mario Kart Wii,2008-01-01,Nintendo,Nintendo,Wii,82.0,8.3,E,Racing
2,sales_3,Wii Sports Resort,2009-01-01,Nintendo,Nintendo,Wii,80.0,8.0,E,Sports
3,sales_4,New Super Mario Bros.,2006-01-01,Nintendo,Nintendo,DS,89.0,8.5,E,Platform
4,sales_5,Wii Play,2006-01-01,Nintendo,Nintendo,Wii,58.0,6.6,E,Misc


In [18]:
# convert genre strings to lists
normalized_sets = [dbpedia_normalized, metacritic_normalized, sales_normalized]
for s in normalized_sets:
    s["genres"] = s["genres"].apply(lambda x: [g.strip() for g in x.split(",")] if isinstance(x, str) else x)


In [19]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
metacritic = metacritic_normalized[metacritic_cols].copy()
sales = sales_normalized[sales_cols].copy()

## Part 2: Data Profiling

In [20]:
# Display basic information
datasets = [dbpedia, metacritic, sales]
names = ["DBpedia", "Metacritic", "Sales"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 74,951


In [21]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

dbpedia:
  Rows: 46,580
  Columns: 7
  Total nulls: 25,449
  Null percentage: 7.8%
  Null counts per column:
    releaseYear: 1,409 (3.0%)
    developer: 1,221 (2.6%)
    platform: 410 (0.9%)
    series: 22,193 (47.6%)
    genres: 216 (0.5%)

metacritic:
  Rows: 20,494
  Columns: 9
  Total nulls: 3,718
  Null percentage: 2.0%
  Null counts per column:
    developer: 19 (0.1%)
    criticScore: 10 (0.0%)
    userScore: 1,413 (6.9%)
    ESRB: 2,276 (11.1%)

sales:
  Rows: 7,877
  Columns: 10
  Total nulls: 1,053
  Null percentage: 1.3%
  Null counts per column:
    publisher: 1 (0.0%)
    userScore: 1,052 (13.4%)



### Attribute Coverage Analysis

In [22]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print(" Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,metacritic_count,metacritic_pct,metacritic_coverage,metacritic_samples,sales_count,sales_pct,sales_coverage,sales_samples,avg_coverage,max_coverage,datasets_with_attribute
0,ESRB,0/0,0%,0.000000,N/A,18218/20494,88.9%,0.888943,"['M', 'M', 'T']",7877/7877,100.0%,1.000000,"['E', 'E', 'E']",0.629648,1.000000,2
1,criticScore,0/0,0%,0.000000,N/A,20484/20494,100.0%,0.999512,"[97.0, 98.0, 98.0]",7877/7877,100.0%,1.000000,"[76.0, 82.0, 80.0]",0.666504,1.000000,2
2,developer,45359/46580,97.4%,0.973787,"['Handheld Games', 'Ocean Software', 'Key (com...",20475/20494,99.9%,0.999073,"['Rockstar Games', 'Rockstar North', 'Namco']",7877/7877,100.0%,1.000000,"['Nintendo', 'Nintendo', 'Nintendo']",0.990953,1.000000,3
3,genres,46364/46580,99.5%,0.995363,"[['Racing video game'], [""Beat 'em up""], ['Ero...",20494/20494,100.0%,1.000000,"[['Action Adventure', 'Open-World'], ['Action ...",7877/7877,100.0%,1.000000,"[['Sports'], ['Racing'], ['Sports']]",0.998454,1.000000,3
4,id,46580/46580,100.0%,1.000000,"['dbpedia_1', 'dbpedia_2', 'dbpedia_3']",20494/20494,100.0%,1.000000,"['metacritic_1', 'metacritic_2', 'metacritic_3']",7877/7877,100.0%,1.000000,"['sales_1', 'sales_2', 'sales_3']",1.000000,1.000000,3
5,name,46580/46580,100.0%,1.000000,"['San Francisco Rush 2049', 'RoboCop (1988 vid...",20494/20494,100.0%,1.000000,"['Red Dead Redemption 2', 'Grand Theft Auto IV...",7877/7877,100.0%,1.000000,"['Wii Sports', 'Mario Kart Wii', 'Wii Sports R...",1.000000,1.000000,3
6,platform,46170/46580,99.1%,0.991198,"['Game Boy Color', 'Arcade video game', 'PlayS...",20494/20494,100.0%,1.000000,"['Xbox One', 'Xbox 360', 'Dreamcast']",7877/7877,100.0%,1.000000,"['Wii', 'Wii', 'Wii']",0.997066,1.000000,3
7,publisher,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7876/7877,100.0%,0.999873,"['Nintendo', 'Nintendo', 'Nintendo']",0.333291,0.999873,1
8,releaseYear,45171/46580,97.0%,0.969751,"[Timestamp('2006-01-01 00:00:00'), Timestamp('...",20494/20494,100.0%,1.000000,"['2018-01-01', '2008-01-01', '1999-01-01']",7877/7877,100.0%,1.000000,"['2006-01-01', '2008-01-01', '2009-01-01']",0.989917,1.000000,3
9,series,24387/46580,52.4%,0.523551,"['Rush (video game series)', 'List of RoboCop ...",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.174517,0.523551,1



 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['ESRB', 'criticScore', 'developer', 'genres', 'id', 'name', 'platform', 'releaseYear', 'userScore']


## Part 3: Entity Matching

### Step 1: Blocking

In [23]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [24]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re
import pandas as pd

TITLE_TOKEN_STOPWORDS = {
    "the", "of", "and", "a", "an", "in", "to", "for", "on", "with", "at", "by", "from",
    "edition", "game", "video", "ii", "iii", "iv"
}

def title_tokens(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name).lower())
    return [
        token for token in tokens
        if len(token) >= 3 and token not in TITLE_TOKEN_STOPWORDS
    ]

def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

platform_aliases = {
    "playstation 4": "ps4", "ps4": "ps4",
    "playstation 3": "ps3", "ps3": "ps3",
    "playstation 2": "ps2", "ps2": "ps2",
    "playstation vita": "ps vita", "ps vita": "ps vita", "psv": "ps vita",
    "playstation portable": "psp", "psp": "psp",
    "xbox one": "xbox one", "xone": "xbox one",
    "xbox 360": "xbox 360", "x360": "xbox 360",
    "xbox": "xbox",
    "nintendo switch": "switch", "switch": "switch",
    "wii": "wii", "wii u": "wii u",
    "gamecube": "gamecube", "nintendo gamecube": "gamecube",
    "nintendo ds": "ds", "ds": "ds",
    "nintendo 3ds": "3ds", "3ds": "3ds",
    "game boy advance": "gba", "gba": "gba",
    "game boy color": "gbc", "gbc": "gbc",
    "pc": "pc", "microsoft windows": "pc", "windows": "pc",
    "macintosh": "mac", "mac os x": "mac",
    "ios": "ios", "android": "android",
    "arcade video game": "arcade", "arcade": "arcade",
}

def normalize_platform_value(value):
    key = str(value).strip().lower()
    return platform_aliases.get(key, key)

def release_year_value(value):
    match = re.search(r"\d{4}", str(value))
    return match.group(0) if match else ""

class UnionTitleTokenBlocker:
    """Union of shared title-token blocks constrained by platform or release year."""

    def __init__(self, df_left, df_right, *, id_column, name_column, platform_column, year_column, batch_size=100_000):
        self.df_left = df_left
        self.df_right = df_right
        self.id_column = id_column
        self.name_column = name_column
        self.platform_column = platform_column
        self.year_column = year_column
        self.batch_size = int(batch_size)
        self.rules = (
            ("title_token_platform", "platform_block_key"),
            ("title_token_year", "release_year_block_key"),
        )
        self._left_blocks = self._build_blocks(df_left)
        self._right_blocks = self._build_blocks(df_right)

    def _build_blocks(self, df):
        blocks = {rule_name: {} for rule_name, _ in self.rules}
        for row in df[[self.id_column, self.name_column, self.platform_column, self.year_column]].itertuples(index=False):
            record_id = getattr(row, self.id_column)
            name = getattr(row, self.name_column)
            platform = normalize_platform_value(getattr(row, self.platform_column))
            release_year = release_year_value(getattr(row, self.year_column))
            for token in title_tokens(name):
                if platform:
                    blocks["title_token_platform"].setdefault((token, platform), []).append(record_id)
                if release_year:
                    blocks["title_token_year"].setdefault((token, release_year), []).append(record_id)
        return blocks

    def __iter__(self):
        seen_pairs = set()
        batch = []
        for rule_name, _ in self.rules:
            left_blocks = self._left_blocks[rule_name]
            right_blocks = self._right_blocks[rule_name]
            for block_key in left_blocks.keys() & right_blocks.keys():
                for left_id in left_blocks[block_key]:
                    for right_id in right_blocks[block_key]:
                        pair = (left_id, right_id)
                        if pair in seen_pairs:
                            continue
                        seen_pairs.add(pair)
                        batch.append((left_id, right_id, f"{rule_name}:{block_key[0]}:{block_key[1]}"))
                        if len(batch) >= self.batch_size:
                            yield pd.DataFrame(batch, columns=["id1", "id2", "block_key"])
                            batch = []
        if batch:
            yield pd.DataFrame(batch, columns=["id1", "id2", "block_key"])

    def materialize(self):
        frames = [batch for batch in self if not batch.empty]
        if frames:
            return pd.concat(frames, ignore_index=True)
        return pd.DataFrame(columns=["id1", "id2", "block_key"])

    def estimate_pairs(self):
        return sum(len(batch) for batch in self)


dbpedia['name_longest_token'] = dbpedia['name'].apply(get_longest_token)
metacritic['name_longest_token'] = metacritic['name'].apply(get_longest_token)
sales['name_longest_token'] = sales['name'].apply(get_longest_token)

standard_blocker_m2d = UnionTitleTokenBlocker(
    dbpedia,
    metacritic,
    id_column='id',
    name_column='name',
    platform_column='platform',
    year_column='releaseYear',
    batch_size=100_000,
)


### Step 2: Evaluate Blocking Against Ground Truth

In [25]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_metacritic_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on the union title-token blocker
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root -   Pair Completeness: 0.991
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   0.999475
[INFO ] root -   True Matches Found: 105/106
[INFO ] root -   Batches Processed:  6
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9905660377358491,
 'pair_quality': 0.0002094779808716678,
 'reduction_ratio': 0.9994749209342466,
 'total_candidates': 501246,
 'total_possible_pairs': 954610520,
 'true_positives_found': 105,
 'total_true_pairs': 106,
 'batches_processed': 6,
 'evaluation_timestamp': '2026-06-09T18:14:09.331169',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/blocking-evaluation/blocking_detailed_results.csv']}

In [26]:
# def get_first_token(name):
#     return re.split(r"[^A-Za-z0-9_']+", str(name))[0]
# metacritic['name_first_token'] = metacritic['name'].apply(get_first_token)
# sales['name_first_token'] = sales['name'].apply(get_first_token)

standard_blocker_m2s = StandardBlocker(
    dbpedia, sales,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_sales_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2s,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4431 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2231 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1731 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 0 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 0 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 0 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 0 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 0 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 1 true matches
[INFO ] root 

{'pair_completeness': 1.0,
 'pair_quality': 0.0003061033309847819,
 'reduction_ratio': 0.9989671681929329,
 'total_candidates': 378957,
 'total_possible_pairs': 366910660,
 'true_positives_found': 116,
 'total_true_pairs': 116,
 'batches_processed': 379,
 'evaluation_timestamp': '2026-06-09T18:14:15.962278',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/blocking-evaluation/blocking_detailed_results.csv']}

Now let's evaluate which blocking method we want to use for each dataset combination:

### Step 3: Entity Matching with Comparators

In [27]:
from PyDI.entitymatching import StringComparator, DateComparator
import difflib

# Create comparators for different attributes
# DBpedia-Metacritic uses a stricter title/platform rule because the high-recall
# blocker intentionally admits many near-title candidates.
def normalize_match_title(value):
    value = str(value).lower()
    value = re.sub(r"\([^)]*video game[^)]*\)", " ", value)
    value = re.sub(
        r"\b(video game|game|hd|remaster(?:ed)?|definitive edition|special edition|complete edition|goty edition)\b",
        " ",
        value,
    )
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

MATCH_TITLE_STOPWORDS = {
    "the", "of", "and", "a", "an", "in", "to", "for", "on", "with", "at", "by", "from",
    "edition", "game", "video"
}

def match_title_tokens(value):
    return {
        token for token in normalize_match_title(value).split()
        if len(token) >= 2 and token not in MATCH_TITLE_STOPWORDS
    }

def name_sequence_similarity(record1, record2):
    return difflib.SequenceMatcher(
        None,
        normalize_match_title(record1["name"]),
        normalize_match_title(record2["name"]),
    ).ratio()

def name_token_overlap_similarity(record1, record2):
    tokens1 = match_title_tokens(record1["name"])
    tokens2 = match_title_tokens(record2["name"])
    if not tokens1 or not tokens2:
        return 0.0
    return len(tokens1 & tokens2) / min(len(tokens1), len(tokens2))

def platform_exact_similarity(record1, record2):
    platform1 = normalize_platform_value(record1["platform"])
    platform2 = normalize_platform_value(record2["platform"])
    return 1.0 if platform1 and platform2 and platform1 == platform2 else 0.0

comparators_m2d = [
    name_sequence_similarity,
    name_token_overlap_similarity,
    platform_exact_similarity,
]

comparators_m2s = [
    StringComparator(
        column='name',
        similarity_function='jaccard',
        preprocess=str.lower
    ),
        # Platform similarity - supporting evidence
    StringComparator(
        column='platform',
        similarity_function='jaccard',
    ),
    DateComparator(
        column='releaseYear',
        max_days_difference=360  # Allow almost 1 year difference
    )
]


Next, we setup the matcher and run the matching with our chosen best blocking method:

In [28]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=dbpedia,
    df_right=metacritic, 
    candidates=standard_blocker_m2d, # union title-token blocker: token+platform OR token+releaseYear
    comparators=comparators_m2d,
    weights=[0.65, 0.25, 0.10], # title sequence, title token overlap, platform
    threshold=0.98, # tuned for the high-recall DBpedia-Metacritic blocker
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 46580 x 20494 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 46580 x 20494 elements after 0:00:0.276; 501246 blocked pairs (reduction ratio: 0.9994749209342466)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:50.243; found 11093 correspondences.


In [29]:
correspondences_m2s = matcher.match(
    df_left=dbpedia,
    df_right=sales, 
    candidates=standard_blocker_m2s,
    comparators=comparators_m2s,
    weights=[0.6, 0.3, 0.1], # name, platform, releaseYear
    threshold=0.8,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 46580 x 7877 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 46580 x 7877 elements after 0:00:0.077; 378957 blocked pairs (reduction ratio: 0.9989671681929329)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:72.685; found 4997 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [30]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_metacritic_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  88
[INFO ] root -   True Negatives:  227
[INFO ] root -   False Positives: 4
[INFO ] root -   False Negatives: 18
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.935
[INFO ] root -   Precision: 0.957
[INFO ] root -   Recall:    0.830
[INFO ] root -   F1-Score:  0.889


{'precision': 0.9565217391304348,
 'recall': 0.8301886792452831,
 'f1': 0.888888888888889,
 'accuracy': 0.9347181008902077,
 'true_positives': 88,
 'false_positives': 4,
 'false_negatives': 18,
 'true_negatives': 227,
 'threshold_used': 0.0,
 'total_correspondences': 11093,
 'filtered_correspondences': 11093,
 'evaluation_timestamp': '2026-06-09T18:16:23.570708',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [31]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 6751 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	4701	|	69.63%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	1183	|	17.52%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	443	|	6.56%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	180	|	2.67%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	91	|	1.35%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	58	|	0.86%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	26	|	0.39%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	26	|	0.39%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	16	|	0.24%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	9	|	0.13%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	1	|	0.01%
[INFO ] PyDI.entitymatching.evaluation - 		13	|	1	|	0.01%
[INFO ] PyDI.entitymatching.evaluatio

Analyzing cluster size distribution in our entity matching results...


In [32]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 6751 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [33]:
from PyDI.entitymatching import MaximumBipartiteMatching
     
clusterer = MaximumBipartiteMatching()
correspondences_m2d_post = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d_post,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d_post,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Filtered correspondences: 11093 -> 11093 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 11093 -> 6778 
[INFO ] root - MaximumBipartiteMatching: 11093 -> 6778 correspondences
[INFO ] root - MaximumBipartiteMatching: 17780 -> 13556 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 6778 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	6778	|	100.00%
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  47
[INFO ] root -   True Negatives:  230
[INFO ] root -   False Positives: 1
[INFO ] root -   False Negatives: 59
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.822
[INFO ] root -   Precision: 0.979
[INFO ] root -   Recall:    0.443
[INFO ] root -   F1-Score:  0.610


In [34]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "dbpedia_2_sales_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

clusterer = MaximumBipartiteMatching()
correspondences_m2s_post = clusterer.cluster(correspondences_m2s)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s_post,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s_post,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)


[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  95
[INFO ] root -   True Negatives:  286
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 21
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.948
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.819
[INFO ] root -   F1-Score:  0.900
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2948 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2045	|	69.37%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	522	|	17.71%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	193	|	6.55%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	74	|	2.51%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	39	|	1.32%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	21	|	0.71%
[INFO ] PyDI.entitymatching.evaluat

## Part 4: Data Fusion

In [35]:
# We are only interested in year of release -> normalize all dates to YYYY-01-01.
# Coerce first so this works whether releaseYear is already datetime or still a string/object column.
def normalize_to_year_start(series):
    dates = pd.to_datetime(series, errors="coerce")
    return dates.dt.to_period("Y").dt.to_timestamp()

for dataset in [dbpedia, metacritic, sales]:
    dataset["releaseYear"] = normalize_to_year_start(dataset["releaseYear"])


In [36]:
metacritic["metacritic_id"] = metacritic["id"]

# Assign trust scores to datasets
metacritic.attrs["trust_score"] = 3
sales.attrs["trust_score"] = 2
dbpedia.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2s], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 16,090


## Step 1: Define Fusion Strategy 

In [37]:
# Identify which attributes are worth fusing:
# attributes that appear in more than one source AND in the fusion groundtruth.
import xml.etree.ElementTree as ET
from collections import defaultdict

sources = {"metacritic": metacritic, "dbpedia": dbpedia, "sales": sales}

# Target schema attributes (skip 'id' since it is the join key, not fused)
target_attrs = [a for a in target_schema["properties"].keys() if a != "id"]

# Groundtruth attributes from the XML directly, so nested elements like
# <genres><genre>...</genre></genres> count as the parent tag "genres".
gt_tree = ET.parse(INPUT_DIR / "fusion" / "validation_set.xml")
gt_attrs = {child.tag for vg in gt_tree.getroot().findall("videogame") for child in vg}

# For each target attribute, which sources actually contain it?
attr_sources = defaultdict(list)
for src_name, df in sources.items():
    for attr in target_attrs:
        if attr in df.columns:
            attr_sources[attr].append(src_name)

rows = []
for attr in target_attrs:
    srcs = attr_sources.get(attr, [])
    rows.append({
        "attribute": attr,
        "num_sources": len(srcs),
        "sources": ", ".join(srcs) if srcs else "—",
        "in_groundtruth": attr in gt_attrs,
        "fuse?": len(srcs) > 1 and attr in gt_attrs,
    })
overview = pd.DataFrame(rows).sort_values(["fuse?", "num_sources"], ascending=[False, False])
display(overview)

fusion_attributes = sorted(overview.loc[overview["fuse?"], "attribute"].tolist())
print("Attributes to fuse:", fusion_attributes)


,attribute,num_sources,sources,in_groundtruth,fuse?
0,name,3,"metacritic, dbpedia, sales",True,True
1,releaseYear,3,"metacritic, dbpedia, sales",True,True
2,developer,3,"metacritic, dbpedia, sales",True,True
3,genres,3,"metacritic, dbpedia, sales",True,True
5,platform,3,"metacritic, dbpedia, sales",True,True
6,criticScore,2,"metacritic, sales",True,True
7,userScore,2,"metacritic, sales",True,True
8,ESRB,2,"metacritic, sales",True,True
4,publisher,1,sales,True,False
9,series,1,dbpedia,False,False


Attributes to fuse: ['ESRB', 'criticScore', 'developer', 'genres', 'name', 'platform', 'releaseYear', 'userScore']


In [38]:
from PyDI.fusion import DataFusionStrategy, prefer_higher_trust, voting, average, union

strategy = DataFusionStrategy('game_fusion_strategy')
['id', 'ESRB', 'criticScore', 'developer', 'genres', 'name', 'platform', 'releaseYear', 'userScore']
strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('platform', voting)
strategy.add_attribute_fuser('developer', voting)
strategy.add_attribute_fuser('releaseYear', voting, trust_key="trust_score")
strategy.add_attribute_fuser('ESRB', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('criticScore', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('userScore', average)
strategy.add_attribute_fuser('genres', union)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'platform' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'developer' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'releaseYear' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'ESRB' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'criticScore' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'userScore' using rule 'average'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'genres' using rule 'union'


## Step 2: Run Fusion

In [39]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[metacritic, dbpedia, sales],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'game_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 21274 of 21274 unique IDs
[INFO ] PyDI.fusion.engine - Created 60649 record groups from 16090 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 60649 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	2962	|	4.88%
[INFO ] PyDI.fusion.engine - 		3	|	2612	|	4.31%
[INFO ] PyDI.fusion.engine - 		4	|	760	|	1.25%
[INFO ] PyDI.fusion.engine - 		5	|	293	|	0.48%
[INFO ] PyDI.fusion.engine - 		6	|	125	|	0.21%
[INFO ] PyDI.fusion.engine - 		7	

Fused rows: 6,972


,_id,_fusion_sources,_fusion_source_datasets,ESRB,criticScore,developer,genres,id,metacritic_id,name,name_longest_token,platform,releaseYear,series,userScore,_fusion_confidence,_fusion_metadata,publisher
0,dbpedia_1456,"[metacritic_1, dbpedia_1456]","[metacritic, dbpedia]",M,97.0,Rockstar Games,"[Action Adventure, Action-adventure, Open-World]",dbpedia_1456,metacritic_1,Red Dead Redemption 2,Redemption,Xbox One,2018-01-01,Red Dead,8.300000,0.772727,"{'ESRB_rule': 'prefer_higher_trust', 'ESRB_sou...",NaN
1,dbpedia_51107,"[metacritic_2, metacritic_20432, dbpedia_51107...","[metacritic, metacritic, dbpedia, sales]",M,98.0,Rockstar North,"[Action, Action Adventure, Action-adventure, M...",dbpedia_51107,metacritic_2,Grand Theft Auto V,Grand,Xbox 360,2013-01-01,Grand Theft Auto,8.133333,0.696639,"{'ESRB_rule': 'prefer_higher_trust', 'ESRB_sou...",Take-Two Interactive
2,dbpedia_9885,"[metacritic_5, dbpedia_9885, sales_32]","[metacritic, dbpedia, sales]",E,97.0,Nintendo,"[3D, Action, Platform, Platform game, Platformer]",dbpedia_9885,metacritic_5,Super Mario Galaxy,Galaxy,Wii,2007-01-01,Super Mario,9.000000,0.804630,"{'ESRB_rule': 'prefer_higher_trust', 'ESRB_sou...",Nintendo
3,dbpedia_44487,"[metacritic_6, metacritic_12, dbpedia_44487, d...","[metacritic, metacritic, dbpedia, dbpedia, dbp...",M,98.0,Rockstar North,"[Action, Action Adventure, Action-adventure, M...",dbpedia_44487,metacritic_12,Grand Theft Auto V,Grand,PS3,2013-01-01,Grand Theft Auto,8.133333,0.706592,"{'ESRB_rule': 'prefer_higher_trust', 'ESRB_sou...",Take-Two Interactive
4,dbpedia_52479,"[metacritic_7, dbpedia_52479, sales_48]","[metacritic, dbpedia, sales]",M,94.0,Infinity Ward,"[Action, Arcade, First-Person, First-person sh...",dbpedia_52479,metacritic_7,Call of Duty 4: Modern Warfare,Warfare,Xbox 360,2007-01-01,Call of Duty,8.450000,0.832840,"{'ESRB_rule': 'prefer_higher_trust', 'ESRB_sou...",Activision


## Step 3: Evaluate Data Fusion

In [40]:
from PyDI.fusion import tokenized_match, year_only_match, boolean_match, numeric_tolerance_match, exact_match, intersection
['genres', 'name', 'platform', 'releaseYear', 'userScore']
strategy.add_evaluation_function("name", exact_match)
strategy.add_evaluation_function("platform", exact_match)
strategy.add_evaluation_function("developer", exact_match)
strategy.add_evaluation_function("releaseYear", year_only_match)
strategy.add_evaluation_function("ESRB", exact_match)
strategy.add_evaluation_function("criticScore", numeric_tolerance_match, tolerance=2)
strategy.add_evaluation_function("userScore", numeric_tolerance_match, tolerance=0.2)
strategy.add_evaluation_function("genres", intersection)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'platform'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'developer'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'releaseYear'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'ESRB'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'criticScore' with params {'tolerance': 2}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'userScore' with params {'tolerance': 0.2}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'genres'


In [41]:
from PyDI.io import load_xml
from PyDI.fusion import DataFusionEvaluator

# Finally, evaluate against validation set
fusion_validation_set = load_xml(INPUT_DIR / 'fusion' / 'validation_set.xml', name='fusion_validation_set', nested_handling='aggregate')
fusion_validation_set['releaseYear'] = pd.to_datetime(fusion_validation_set['releaseYear'],errors='coerce')
fusion_validation_set = fusion_validation_set.rename(columns={'genres_genre': 'genres'})

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the validation set
print("Evaluating fusion results against validation set...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='metacritic_id',
    gold_df=fusion_validation_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion validation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[WARNING] PyDI.fusion.evaluation - Missing 21 expected/reference records in fused dataset: metacritic_238, metacritic_1695, metacritic_336, metacritic_474, metacritic_447, ...
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.703 overall accuracy (499/710)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 211 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	userScore                        |      58 |     27.49%%
[INFO ] PyDI.fusion.evaluation - 	genres                           |      51 |     24.17%%
[INFO ] PyDI.fusion.eva

Evaluating fusion results against validation set...

Fusion validation Results:
  overall_accuracy: 0.703
  macro_accuracy: 0.702
  num_evaluated_records: 79
  num_evaluated_attributes: 9
  total_evaluations: 710
  total_correct: 499
  developer_accuracy: 0.911
  developer_count: 79
  platform_accuracy: 0.975
  platform_count: 79
  userScore_accuracy: 0.256
  userScore_count: 78
  publisher_accuracy: 0.418
  publisher_count: 79
  name_accuracy: 0.949
  name_count: 79
  criticScore_accuracy: 0.570
  criticScore_count: 79
  genres_accuracy: 0.354
  genres_count: 79
  releaseYear_accuracy: 0.886
  releaseYear_count: 79
  ESRB_accuracy: 1.000
  ESRB_count: 79

Overall Accuracy: 70.3%


In [42]:
from PyDI.io import load_xml
from PyDI.fusion import DataFusionEvaluator

# Finally, evaluate against test set
fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')
fusion_test_set['releaseYear'] = pd.to_datetime(fusion_test_set['releaseYear'],errors='coerce')
fusion_test_set = fusion_test_set.rename(columns={'genres_genre': 'genres'})

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the test set
print("Evaluating fusion results against test set...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='metacritic_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Test Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/games/output/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation


Evaluating fusion results against test set...


[WARNING] PyDI.fusion.evaluation - Missing 23 expected/reference records in fused dataset: metacritic_15553, metacritic_8409, metacritic_16320, metacritic_3855, metacritic_16979, ...
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.718 overall accuracy (497/692)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 195 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	userScore                        |      56 |     28.72%%
[INFO ] PyDI.fusion.evaluation - 	genres                           |      45 |     23.08%%
[INFO ] PyDI.fusion.evaluation - 	publisher                        |      36 |     18.46%%
[INFO ] PyDI.fusion.evaluation - 	criticScore                      |      33 |     16.92%%
[INFO ] PyDI.fusion.evaluation - 	developer                        |      10 |      5.13%%
[INFO ] 


Fusion Test Results:
  overall_accuracy: 0.718
  macro_accuracy: 0.718
  num_evaluated_records: 77
  num_evaluated_attributes: 9
  total_evaluations: 692
  total_correct: 497
  developer_accuracy: 0.870
  developer_count: 77
  platform_accuracy: 0.948
  platform_count: 77
  userScore_accuracy: 0.263
  userScore_count: 76
  publisher_accuracy: 0.532
  publisher_count: 77
  name_accuracy: 0.935
  name_count: 77
  criticScore_accuracy: 0.571
  criticScore_count: 77
  genres_accuracy: 0.416
  genres_count: 77
  releaseYear_accuracy: 0.935
  releaseYear_count: 77
  ESRB_accuracy: 0.987
  ESRB_count: 77

Overall Accuracy: 71.8%


In [43]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal execution time: {elapsed_time:.2f} seconds")


Total execution time: 179.78 seconds
